In [ ]:
# ============================================================
# Cell 0: 硬件自检 —— 概念章，CPU 即可
# ============================================================
# 本章在一维二次函数 + 1000 个 2D 点上演示优化器与调度，CPU 完全够用。
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device         :', torch.cuda.get_device_name(0))
else:
    print('未检测到 GPU —— 本章 CPU 完全够用，继续执行即可。')


In [ ]:
%%capture
# ============================================================
# Cell 1: 安装/升级依赖库
# ============================================================
# 重要：%%capture 必须是 cell 第一行，否则会被当 line magic 报错。
# scikit-learn: make_moons 玩具数据
# matplotlib:   画优化轨迹、lr 曲线、loss 曲线
# torch 在 Colab 已预装。
!pip install -q -U scikit-learn matplotlib


In [ ]:
# ============================================================
# Cell 2: SGD vs Momentum vs Adam —— 在一个二次函数上画轨迹
# ============================================================
# 目标函数：f(x, y) = 0.5 * (x^2 + 10 * y^2) —— 长椭圆山谷
# 不同方向曲率差 10 倍，纯 SGD 会沿 y 方向震荡。
import torch
import matplotlib.pyplot as plt

def loss_fn(p):
    return 0.5 * (p[0] ** 2 + 10 * p[1] ** 2)

def run(optimizer_cls, kwargs, lr, n_steps=40, init=(-3.0, 1.5)):
    p = torch.tensor(init, requires_grad=True)
    opt = optimizer_cls([p], lr=lr, **kwargs)
    traj = [p.detach().clone().tolist()]
    for _ in range(n_steps):
        loss = loss_fn(p)
        opt.zero_grad()
        loss.backward()
        opt.step()
        traj.append(p.detach().clone().tolist())
    return traj

trajs = {
    'SGD lr=0.05':           run(torch.optim.SGD,  {},                              lr=0.05),
    'SGD+Momentum lr=0.05':  run(torch.optim.SGD,  {'momentum': 0.9},               lr=0.05),
    'Adam lr=0.3':           run(torch.optim.Adam, {'betas': (0.9, 0.999)},         lr=0.3),
}

# 画等高线 + 三条优化轨迹
import numpy as np
xs = np.linspace(-3.5, 0.5, 200)
ys = np.linspace(-2.0, 2.0, 200)
XX, YY = np.meshgrid(xs, ys)
ZZ = 0.5 * (XX ** 2 + 10 * YY ** 2)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, traj) in zip(axes, trajs.items()):
    ax.contour(XX, YY, ZZ, levels=20, cmap='Greys', alpha=0.5)
    traj = np.array(traj)
    ax.plot(traj[:, 0], traj[:, 1], '-o', markersize=3)
    ax.scatter([0], [0], marker='*', color='red', s=100, label='optimum')
    ax.set_title(name)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print('观察：')
print('  纯 SGD —— 沿 y 方向（曲率大）来回震荡，沿 x（曲率小）走得慢')
print('  +Momentum —— 震荡明显减弱，沿主方向加速')
print('  Adam —— 自适应步长几乎走直线，是三种里最"无脑稳"的')


In [ ]:
# ============================================================
# Cell 3: 手写一遍 Adam，验证与 PyTorch 内置一致
# ============================================================
# Adam 的更新公式（含 bias correction）：
#   m ← β1 m + (1-β1) g
#   v ← β2 v + (1-β2) g^2
#   m̂ = m / (1 - β1^t),  v̂ = v / (1 - β2^t)
#   θ ← θ - lr · m̂ / (√v̂ + eps)
import torch

torch.manual_seed(0)
# 共享初值 + 同样的"梯度序列"，看两种实现是否一致
def make_grads():
    return [torch.randn(3) for _ in range(20)]

grads = make_grads()
lr, b1, b2, eps = 1e-2, 0.9, 0.999, 1e-8

# (a) PyTorch 内置 Adam
theta_a = torch.zeros(3, requires_grad=True)
opt = torch.optim.Adam([theta_a], lr=lr, betas=(b1, b2), eps=eps)
for g in grads:
    opt.zero_grad()
    theta_a.grad = g.clone()                       # 直接灌入梯度，绕过 backward
    opt.step()

# (b) 手写 Adam
theta_b = torch.zeros(3)
m = torch.zeros(3); v = torch.zeros(3)
for t, g in enumerate(grads, start=1):
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * g * g
    m_hat = m / (1 - b1 ** t)
    v_hat = v / (1 - b2 ** t)
    theta_b = theta_b - lr * m_hat / (v_hat.sqrt() + eps)

print('PyTorch Adam   :', theta_a.detach().tolist())
print('手写  Adam     :', theta_b.tolist())
print('两种实现一致 :', torch.allclose(theta_a.detach(), theta_b, atol=1e-6))


In [ ]:
# ============================================================
# Cell 4: lr 太大 / 合适 / 太小 —— 同一个模型同一份数据三种结果
# ============================================================
# 用 P02 的 make_moons 做演示，模型固定为 2 → 16 → 1 的 ReLU MLP。
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=0)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32)

def train_with_lr(lr, n_epoch=200, seed=0):
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    losses = []
    for _ in range(n_epoch):
        logits = model(X).squeeze(-1)
        loss = F.binary_cross_entropy_with_logits(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

# 三种 lr：太大可能 NaN / 振荡，太小慢得离谱
lrs = [5.0, 0.1, 0.001]
plt.figure(figsize=(7, 4))
for lr in lrs:
    losses = train_with_lr(lr)
    plt.plot(losses, label=f'lr = {lr}')
plt.xlabel('epoch'); plt.ylabel('loss')
plt.title('Same model, only lr varies: lr is the most sensitive hyperparameter')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print('观察：')
print('  lr=5.0  —— loss 振荡甚至爆炸；步子太大撞过山谷')
print('  lr=0.1  —— 平稳收敛到低位；这是合适的范围')
print('  lr=0.001 —— 下降但慢得离谱；同样的 epoch 远未收敛')


In [ ]:
# ============================================================
# Cell 5: 把 warmup + cosine 调度的 lr 曲线画出来
# ============================================================
# 写法上 PyTorch 里有两种：
#   (a) 用 LambdaLR 自己写 lr_lambda（最直观，便于理解）
#   (b) 用 transformers 的 get_cosine_schedule_with_warmup（生产实操）
# 这里展示 (a)，把"调度本质上就是把 step 映射成 lr 倍率"看透。
import math
import torch
import matplotlib.pyplot as plt

peak_lr = 1.0       # 用 1.0 方便看比例
T_w = 200           # warmup step 数
T   = 2000          # 总 step 数
eta_min_ratio = 0.1  # cosine 退火下界 = 10% peak

def lr_lambda(step):
    # 返回 lr 相对 peak 的"倍率"
    if step < T_w:
        # 线性 warmup：0 → 1
        return step / T_w
    # cosine decay：1 → eta_min_ratio
    progress = (step - T_w) / (T - T_w)
    return eta_min_ratio + 0.5 * (1 - eta_min_ratio) * (1 + math.cos(math.pi * progress))

steps = list(range(T))
lrs = [peak_lr * lr_lambda(s) for s in steps]

plt.figure(figsize=(8, 3.5))
plt.plot(steps, lrs)
plt.axvline(T_w, color='gray', linestyle='--', label=f'warmup end (step={T_w})')
plt.axhline(peak_lr * eta_min_ratio, color='green', linestyle=':', label=f'η_min = {eta_min_ratio:.0%} peak')
plt.xlabel('step'); plt.ylabel('lr')
plt.title('warmup + cosine: default LR schedule shape for LLM training')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

# 真实训练里就把 lr_lambda 喂给 LambdaLR：
# scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
# 训练循环里  optimizer.step()  之后调用  scheduler.step()
print('注意：scheduler.step() 必须在 optimizer.step() 之后调，否则 lr 会推迟一步生效。')


In [ ]:
# ============================================================
# Cell 6: 综合训练对比 —— SGD / SGD+Momentum / Adam / AdamW
# ============================================================
# 把 P02 的 make_moons 二分类延续过来，比四种优化器的 loss 曲线。
# 模型、初始化、数据完全一致；区别只在优化器与 lr。
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=0)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32)

def make_model(seed=42):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))

def train(opt_cls, opt_kwargs, n_epoch=200):
    model = make_model()
    opt = opt_cls(model.parameters(), **opt_kwargs)
    losses = []
    for _ in range(n_epoch):
        logits = model(X).squeeze(-1)
        loss = F.binary_cross_entropy_with_logits(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

results = {
    'SGD lr=0.1':                    train(torch.optim.SGD,   {'lr': 0.1}),
    'SGD+Momentum lr=0.1, μ=0.9':    train(torch.optim.SGD,   {'lr': 0.1, 'momentum': 0.9}),
    'Adam lr=1e-2':                  train(torch.optim.Adam,  {'lr': 1e-2}),
    'AdamW lr=1e-2, wd=1e-2':        train(torch.optim.AdamW, {'lr': 1e-2, 'weight_decay': 1e-2}),
}

plt.figure(figsize=(8, 4))
for name, losses in results.items():
    plt.plot(losses, label=name)
plt.xlabel('epoch'); plt.ylabel('loss')
plt.title('Training curves of four optimizers on make_moons')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print('观察（玩具数据上不一定碾压，但趋势成立）：')
print('  SGD —— 收敛最慢；非凸地形里是最朴素的基线')
print('  +Momentum —— 比纯 SGD 显著快')
print('  Adam —— 自适应步长，前期下降最猛')
print('  AdamW —— 与 Adam 相近，但 weight decay 是 LLM 长训练时稳定的关键')


In [ ]:
# ============================================================
# Cell 7: warmup 的实际效果 —— 故意"无 warmup vs 有 warmup"
# ============================================================
# 这里用一个故意"难起步"的设置：把模型初始化拉大、用偏大的 lr。
# 对比直接用 peak lr 与先 warmup 再用 peak lr 的 loss 曲线。
import math
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=0)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32)

def make_model(seed=0):
    torch.manual_seed(seed)
    m = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 1))
    # 故意把初始权重放大 5 倍，让起步更"颠簸"
    with torch.no_grad():
        for p in m.parameters():
            p.mul_(5.0)
    return m

def train(use_warmup, peak_lr=0.5, n_epoch=300, T_w=50):
    model = make_model()
    opt = torch.optim.SGD(model.parameters(), lr=peak_lr)
    if use_warmup:
        # 前 T_w step 线性升、之后保持 peak
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lr_lambda=lambda s: min(1.0, (s + 1) / T_w))
    losses = []
    for ep in range(n_epoch):
        logits = model(X).squeeze(-1)
        loss = F.binary_cross_entropy_with_logits(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
        if use_warmup: sched.step()
        losses.append(loss.item())
    return losses

losses_no = train(use_warmup=False)
losses_wm = train(use_warmup=True)

plt.figure(figsize=(8, 4))
plt.plot(losses_no, label='no warmup')
plt.plot(losses_wm, label='with warmup (50 steps)')
plt.xlabel('epoch'); plt.ylabel('loss')
plt.title('Effect of warmup: most visible with large init + large lr')
plt.ylim(0, max(max(losses_no), 2.0))
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print('观察：无 warmup 时开头 loss 飙到很高甚至发散；')
print('     有 warmup 时前 50 步小步走，参数已先"找到下山方向"，之后再用 peak lr 才稳。')


In [ ]:
# ============================================================
# Cell 8: 真实训练循环里 warmup + cosine 怎么写
# ============================================================
# 这是把 P02 的训练循环骨架与 P04 的调度拼起来的"成品"模板。
# 后续读 LLM 训练脚本时，下面这套骨架几乎一字不差地能套上。
import math
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import make_moons

X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=0)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32)

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 1))
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-2,                # peak lr
    betas=(0.9, 0.95),      # β2 比 0.999 略小，对早期更敏感（LLM 常见配置）
    weight_decay=1e-2,
)

# warmup + cosine schedule
T_w = 50
T   = 500
eta_min_ratio = 0.1

def lr_lambda(step):
    if step < T_w:
        return step / T_w                         # linear warmup
    progress = (step - T_w) / (T - T_w)
    return eta_min_ratio + 0.5 * (1 - eta_min_ratio) * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

losses, lrs = [], []
for step in range(T):
    logits = model(X).squeeze(-1)
    loss = F.binary_cross_entropy_with_logits(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()                              # ① 先 optimizer.step
    scheduler.step()                              # ② 再 scheduler.step（绝不能反过来）

    losses.append(loss.item())
    lrs.append(optimizer.param_groups[0]['lr'])   # 当前 lr 直接从 param_groups 读

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(lrs); axes[0].set_xlabel('step'); axes[0].set_ylabel('lr')
axes[0].set_title('Actual lr vs step (warmup + cosine)')
axes[0].axvline(T_w, color='gray', linestyle='--')
axes[1].plot(losses); axes[1].set_xlabel('step'); axes[1].set_ylabel('loss')
axes[1].set_title('Training loss curve')
plt.tight_layout()
plt.show()

print('要点回顾：')
print('  - optimizer.zero_grad() / loss.backward() / optimizer.step() 三件套不变')
print('  - scheduler.step() 在 optimizer.step() 之后调用')
print('  - 当前 lr 始终能从 optimizer.param_groups[0]["lr"] 读到，方便 logging')
